# Results

Within this notebook, we load the results, and create the tables and plots for the paper. 

In [ ]:
# we run the notebook from the top-level of the repo, uncomment the next line
%cd /Users/stefan/Documents/Studium-Dateien-und-Blaetter/Bachelorarbeit/sortbench_git/sortbench

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

def plot_metrics(df_results, mode, metric_name, benchmark_type, file=None):
    if mode == 'basic':
        type_names = ['Int-0:1000', 'Float-0:1000', 'English']
        
        _, axes = plt.subplots(2, 2, figsize=(6, 5))
        for i, type in enumerate(type_names):
            row = i%2
            col = i//2
            ax = axes[row, col]
            df_results_type = df_results[df_results['Type'] == type]
            sns.lineplot(x='Size', y=metric_name, data=df_results_type, hue='Model', style='Model', errorbar=None, ax=ax)
            ax.set_title(f'{type}')
            ax.set_ylabel(f'{metric_name}')
        plt.suptitle(f'{benchmark_type} {metric_name}')
        plt.tight_layout()
        axes[1,1].legend(*axes[1,0].get_legend_handles_labels(), loc='upper left')
        axes[1,1].axis('off')
        axes[0,0].legend().remove()
        axes[0,1].legend().remove()
        axes[1,0].legend().remove()
    elif mode == 'advanced':
        type_names = ['Int-10000000:10001000',
                      'Int-n1000:1000',
                      'Float-10000000:10001000',
                      'Float-0:0.0001',
                      'Float-n1000-1000',
                      'ascii',
                      'AsCiI',
                      'PrfxEnglish',
                      'NumberWords']
        
        _, axes = plt.subplots(5, 2, figsize=(8, 14))
        for i, type in enumerate(type_names):
            row = i%5
            col = i//5
            ax = axes[row, col]
            df_results_type = df_results[df_results['Type'] == type]
            sns.lineplot(x='Size', y=metric_name, data=df_results_type, hue='Model', style='Model', errorbar=None, ax=ax)
            ax.set_title(f'{type}')
            ax.set_ylabel(f'{metric_name}')
        plt.suptitle(f'{benchmark_type} {metric_name}')
        plt.tight_layout()
        axes[4,1].legend(*axes[4,0].get_legend_handles_labels(), loc='upper left')
        axes[4,1].axis('off')
        axes[0,0].legend().remove()
        axes[0,1].legend().remove()
        axes[1,0].legend().remove()
        axes[1,1].legend().remove()
        axes[2,0].legend().remove()
        axes[2,1].legend().remove()
        axes[3,0].legend().remove()
        axes[3,1].legend().remove()
        axes[4,0].legend().remove()
    elif mode == 'debug':
        type_names = ['Int-Sorted', 'Float-Sorted', 'English-Sorted', 'Int-Duplicate', 'Float-Duplicate', 'English-Duplicate']
        
        _, axes = plt.subplots(3, 2, figsize=(6, 7))
        for i, type in enumerate(type_names):
            row = i%3
            col = i//3
            ax = axes[row, col]
            df_results_type = df_results[df_results['Type'] == type]
            sns.lineplot(x='Size', y=metric_name, data=df_results_type, hue='Model', style='Model', errorbar=None, ax=ax)
            ax.set_title(type)
        plt.suptitle(metric_name)
        plt.tight_layout()
        axes[2,0].legend(ncol=2, loc='lower left', bbox_to_anchor=(0.3, -0.9))
        axes[0,0].legend().remove()
        axes[0,1].legend().remove()
        axes[1,0].legend().remove()
        axes[1,1].legend().remove()
        axes[2,1].legend().remove()

    if file is not None:
        plt.savefig(file, dpi=300, bbox_inches='tight')
    plt.show()

def plot_aggregated_results(df_results, mode, benchmark_type, file=None):

    metrics = ['Overall Score', 'Validity Score', 'Benchmark Score', 'Faithfulness Score']
    if benchmark_type in ['Any', 'All']:
        metrics = ['Overall Score', 'Validity Score', 'Benchmark Score']
    _, axes = plt.subplots(2, 2, figsize=(9,6))
    for i, metric in enumerate(metrics):
        ax = axes[i//2, i%2]
        if mode=='all':
            df_results_mode = df_results
        else:
            df_results_mode = df_results[df_results['Mode'] == mode]
        sns.lineplot(x='Size', y=metric, data=df_results_mode, hue='Model', style='Model', ax=ax, errorbar=None)
        ax.set_title(f"{metric} {benchmark_type} ({mode})")
    
    plt.tight_layout()
    axes[0,0].legend().remove()
    axes[0,1].legend().remove()
    axes[1,1].legend().remove()
    axes[1,0].legend(ncol=3, loc='lower left', bbox_to_anchor=(0.6, -0.6))
    
    if file is not None:
        plt.savefig(file, dpi=300, bbox_inches='tight')
    plt.show()

def aggregate_scores(df_results, mode):
    if mode=='all':
        df_results_mode = df_results
    else:
        df_results_mode = df_results[df_results['Mode'] == mode]
    df_results_mode = df_results_mode[['Model', 'Size', 'Overall Score', 'Benchmark Score', 'Faithfulness Score', 'Validity Score']]

    results_per_size = df_results_mode.groupby(['Model', 'Size'], observed=True).mean(numeric_only=True).reset_index()
    sum_of_sizes = results_per_size['Size'].unique().sum()
    results_per_size['Weighted Overall Score'] = results_per_size['Size']*results_per_size['Overall Score']
    results_per_size['Weighted Benchmark Score'] = results_per_size['Size']*results_per_size['Benchmark Score']
    results_per_size['Weighted Validity Score'] = results_per_size['Size']*results_per_size['Validity Score']
    results_per_size['Weighted Faithfulness Score'] = results_per_size['Size']*results_per_size['Faithfulness Score']
    
    score_cols = [
        'Weighted Overall Score',
        'Weighted Benchmark Score',
        'Weighted Faithfulness Score',
        'Weighted Validity Score'
    ]

    results_per_model = (results_per_size.groupby('Model', observed=True)[score_cols].sum(numeric_only=True) / sum_of_sizes)
    #results_per_model = results_per_size.groupby('Model', observed=True).sum(numeric_only=True)[['Weighted Overall Score', 'Weighted Benchmark Score', 'Weighted Faithfulness Score', 'Weighted Validity Score']]/sum_of_sizes
    results_per_model = pd.DataFrame(results_per_model).sort_values('Weighted Overall Score', ascending=False).reset_index()
    return results_per_model

def create_latex_table(results_per_model, mode):
    latex_code = results_per_model.to_latex(index=False, float_format="%.3f")
    latex_code = latex_code.replace('Weighted ', '')
    rpl_string = r'\midrule\n\parbox[t]{2mm}{\multirow{7}{*}{\rotatebox[origin=c]{90}{'+mode.title()+r'}}}'
    latex_code = latex_code.replace(r'\midrule', rpl_string)
    return latex_code

def adjust_names_for_output(df_results):
    model_name_map = {'gpt-4o': 'GPT-4o',
                      'gpt-4o-mini': 'GPT-4o-mini',
                      'gpt-5-mini_reasoning_minimal': 'GPT-5-mini',
                      'gpt-5.1_reasoning_none': 'GPT-5.1',
                      'gpt-5.1_reasoning_low': 'GPT-5.1-RL',
                      'claude-3-5-haiku-20241022': 'Claude-3.5-Haiku',
                      'claude-3-5-sonnet-20241022': 'Claude-3.5-Sonnet',
                      'claude-sonnet-4-5-20250929_reasoning_disabled': 'Claude-4.5-Sonnet',
                      'claude-haiku-4-5-20251001_reasoning_disabled': 'Claude-4.5-Haiku',
                      'claude-haiku-4-5-20251001_reasoning_enabled': 'Claude-4.5-Haiku-RE',
                      'claude-sonnet-4-5-20250929_reasoning_enabled': 'Claude-4.5-Sonnet-RE',
                      "gemini-3-pro-preview_reasoning_low": "Gemini-3-pro-RL",
                      "gemini-3-flash-preview_reasoning_none": "Gemini-3-flash",
                      'llama3.1': 'Llama-3.1',
                      'deepseekr1': 'DeepSeek-r1'
                    }
    df_results['Model'] = df_results['Model'].replace(model_name_map)
    # ['Claude-3.5-Haiku', 'Claude-3.5-Sonnet', 'Claude-4.5-Sonnet', 'Claude-4.5-Haiku', 'GPT-4o', 'GPT-4o-mini', 'GPT-5-mini', 'GPT-5.1', 'GPT-5.1-RL', 'Llama-3.1', 'DeepSeek-r1', 'o3-mini']
    df_results['Model'] = pd.Categorical(df_results['Model'], ['Claude-4.5-Sonnet', 'Claude-4.5-Sonnet-RE', 'Claude-4.5-Haiku', 'Claude-4.5-Haiku-RE', 'GPT-5-mini', 'GPT-5.1', 'GPT-5.1-RL', 'Gemini-3-flash', 'Gemini-3-pro-RL'])

    type_name_map = {'ASCII-Lower': 'ascii',
                     'ASCII-Cased': 'AsCiI',
                     'Integers-10000000:10001000': 'Int-10000000:10001000'}
    df_results['Type'] = df_results['Type'].replace(type_name_map)

    benchmark_type_name_map = {'sort-descending': 'Sort Descending',
                     'sort': 'Sort Ascending',
                     'filter-lower': 'Filter Lower',
                     'filter-higher': 'Filter Higher',
                     'any': 'Any',
                     'all': 'All'}
    df_results['Benchmark Type'] = df_results['Benchmark Type'].replace(benchmark_type_name_map)
    df_results['Benchmark Type'] = pd.Categorical(df_results['Benchmark Type'], ['Sort Ascending', 'Sort Descending', 'Filter Lower', 'Filter Higher', 'Any', 'All'])
    return df_results

In [ ]:
benchmark_modes = ['basic', 'advanced'] #['basic', 'advanced', 'debug']
benchmark_types = ['Sort Ascending', 'Sort Descending', 'Filter Lower', 'Filter Higher', 'Any', 'All']
version = "v2.0"

results = {}
for mode in benchmark_modes:
    df_results = pd.read_csv(f'./test_scores/scores_{mode}_{version}.csv')
    df_results = adjust_names_for_output(df_results)
    results[mode] = df_results[df_results['Mode'] == mode]
    for benchmark_type in benchmark_types:
        temp_df_results = df_results[df_results['Benchmark Type'] == benchmark_type]
        results_per_model = aggregate_scores(temp_df_results, mode)
        benchmark_type_str = benchmark_type.replace(" ", "-").lower()
        print(create_latex_table(results_per_model, mode))
        # plot_aggregated_results(temp_df_results, mode, benchmark_type, file=f'./figures/{mode}_{benchmark_type_str}_aggregated_results.pdf')
        # plot_metrics(temp_df_results, mode, 'Overall Score', benchmark_type, file=f'./figures/{mode}_{benchmark_type_str}_overall_score.pdf')
        # plot_metrics(temp_df_results, mode, 'Benchmark Score', benchmark_type, file=f'./figures/{mode}_{benchmark_type_str}_benchmark_score.pdf')
        # if benchmark_type not in ['Any', 'All']:
        #     plot_metrics(temp_df_results, mode, 'Faithfulness Score', benchmark_type, file=f'./figures/{mode}_{benchmark_type_str}_faithfulness_score.pdf')
        # plot_metrics(temp_df_results, mode, 'Validity Score', benchmark_type, file=f'./figures/{mode}_{benchmark_type_str}_validity_score.pdf')

# results for all tasks together
df_results_all = pd.concat(results.values(), ignore_index=True)
results_per_model_all = aggregate_scores(df_results_all, 'all')
print(create_latex_table(results_per_model_all, 'all'))

In [ ]:
error_order = df_results_all.groupby(['ErrorType']).count()['Model'].sort_values(ascending=False).index.tolist()
df_results_all['ErrorType'] = pd.Categorical(df_results_all['ErrorType'], error_order)

fig, axes = plt.subplots(2,1, figsize=(6, 4), gridspec_kw={'height_ratios': [1, 5]})
ax = axes[0]
sns.histplot(df_results_all[(df_results_all['ErrorType']=='Missing closing bracket' ) | (df_results_all['ErrorType']=='Content after last closing bracket') | (df_results_all['ErrorType']=='Extra characters before boolean') | (df_results_all['ErrorType']=='Extra characters after boolean')], y='ErrorType', hue='Model', multiple='stack', legend=False, ax=ax)
ax.set_xlabel(None)
ax.set_ylabel(None)
ax = axes[1]
df_results_all_filtered = df_results_all.loc[(df_results_all['ErrorType']!='Missing closing bracket' ) & (df_results_all['ErrorType']!='Content after last closing bracket') & (df_results_all['ErrorType']!='Extra characters before boolean') & (df_results_all['ErrorType']!='Extra characters after boolean'), :].copy()
df_results_all_filtered['ErrorType'] = pd.Categorical(df_results_all_filtered['ErrorType'], [e for e in error_order if e not in ['Missing closing bracket','Content after last closing bracket']])
sns.set_palette("colorblind")
sns.histplot(df_results_all_filtered, y='ErrorType', hue='Model', multiple='stack', ax=ax, legend=False)
ax.set_xticks(range(0, 21, 4))

models = df_results_all['Model'].unique()
colors = sns.color_palette(n_colors=len(models))

# Handles und Labels selber bauen
from matplotlib.patches import Patch
handles = [Patch(color=colors[i], label=models[i]) for i in range(len(models))]

fig.legend(handles, labels=models, loc='upper center', bbox_to_anchor=(0.55, 0.45), ncol=2)
plt.ylabel('List Type')
plt.suptitle('Types of non-python outputs we could parse')
plt.savefig('figures/error_types:v2.0.pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
melted_results_thinking = pd.melt(df_results_all, id_vars=['Model', 'Validity Score', 'Benchmark Score', 'Faithfulness Score', 'Overall Score'], value_vars=['Thinking Length'], var_name='Metric', value_name='Value')

thinking_models = ["Claude-4.5-Haiku-RE", "Claude-4.5-Sonnet-RE", "GPT-5.1-RL", "Gemini-3-pro-RL"]

filtered_thinking = df_results_all[df_results_all['Model'].isin(thinking_models)]
filtered_thinking = pd.melt(filtered_thinking, id_vars=['Model', 'Size', 'Benchmark Type', 'List Name'], value_vars=['Thinking Length'], value_name='Thiking Length').drop(columns=['Size', 'Benchmark Type', 'List Name', 'variable'])

for model in thinking_models:
    plt.figure(figsize=(6, 3))
    sns.boxplot(data=melted_results_thinking[melted_results_thinking['Model']==model], x='Validity Score', y='Value', showfliers=False, color='lightgray')
    plt.title(f'Thinking Length for {model}')
    plt.ylabel('Number of Thinking Tokens')
    plt.savefig(f'figures/thinking_length_{model}.pdf', bbox_inches='tight')
    plt.show()
    plt.figure(figsize=(6, 3))
    sns.scatterplot(data=melted_results_thinking[melted_results_thinking['Model']==model], x='Benchmark Score', y='Value')
    plt.yscale('log')
    plt.show()
    plt.figure(figsize=(6, 3))
    sns.scatterplot(data=melted_results_thinking[melted_results_thinking['Model']==model], x='Faithfulness Score', y='Value')
    plt.yscale('log')
    plt.show()

In [ ]:
# disable warnings
import warnings
warnings.filterwarnings('ignore')

from autorank import autorank, latex_table

# convert to wide form
for size in df_results_all['Size'].unique():
    results_wide = df_results_all.loc[df_results_all['Size']==size, ['Model', 'Type', 'Benchmark Type', 'Size', 'List Name', 'Benchmark Score']].pivot(index=['Benchmark Type', 'Type', 'Size', 'List Name'], columns='Model', values='Benchmark Score')
    results_wide.reset_index(inplace=True, drop=True)
    res = autorank(results_wide, approach='bayesian', order='descending')
    
    print('% list size:', size)
    latex_table(res)
    print()
    display(res.posterior_matrix)